In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from global_model_periodic_energy_20 import LearnedSimulator_periodic
import torch.nn as nn
import torch.optim as optim
from scipy.spatial import Voronoi

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision('high')
normalization_stats = {
    "velocity": {"mean": torch.tensor([0.0, 0.0]).to(device), "std": torch.tensor([1e-3, 1e-3]).to(device)},
    "acceleration": {"mean": torch.tensor([0.0, 0.0]).to(device), "std": torch.tensor([1e0, 1e0]).to(device)}
}

In [3]:
checkpoint = torch.load("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/GNN for energy/GNN_20cells_8knn/Energy_with_perimeter_20cell_knn8_512_exp1/model_195.pth")
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint.items()}
model = LearnedSimulator_periodic(num_dimensions=2, normalization_stats=normalization_stats, device=device,n_cells = 20)
model.load_state_dict(new_state_dict)
model.to(device)

LearnedSimulator_periodic(
  (graph_network): EnergyGNN(
    (edge_to_node): edgeToNode()
    (gnn_layer1): NodeGNN()
    (gnn_layer2): NodeGNN()
    (gnn_layer3): NodeGNN()
    (gnn_layer4): NodeGNN()
    (gnn_layer5): NodeGNN()
    (regressor): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=1, bias=True)
    )
  )
)

In [4]:
n_cells = 20
n_steps = 200   # nombre d'étapes simulées
dt = 0.1  # pas de temps
max_iter_per_step = 100  # iterations internes de LBFGS par step
lr = 0.1

In [5]:
df = pd.read_csv("/home/jeanlienhard/Documents/Cell_GNN/Data/raw_data/positions_21.csv")
x0 = df[df["step"] == 0].iloc[:n_cells][['x', 'y']].values.astype(np.float32)

# positions initiales comme variable optimisable
positions = torch.tensor(x0, dtype=torch.float32, device=device, requires_grad=True)
prev_prev_positions = positions.clone().detach()
prev_positions = positions.clone().detach()
# pour stocker la trajectoire
trajectory = [positions.detach().cpu().numpy()]

# vitesse initiale nulle
prev_positions = positions.clone().detach()

In [6]:
def make_periodic_copies(x_centers):
    x_offset, y_offset = 2.0, 2.0
    copies = [x_centers]
    for gx in range(-2,3):
        for gy in range(-2,3):
            if gx != 0 or gy != 0:
                shift = torch.tensor([gx*x_offset, gy*y_offset], device=x_centers.device)
                copies.append(x_centers + shift)
    x_full = torch.cat(copies, dim=0)
    return x_full.unsqueeze(1)


In [7]:
optimizer = torch.optim.LBFGS([positions], lr=lr, max_iter=max_iter_per_step, history_size=10, line_search_fn="strong_wolfe")

for step in range(n_steps):
    def closure():
        optimizer.zero_grad()
        x_centers = positions
        
        x_full = make_periodic_copies(x_centers)
        E_cell = model(x_full, n_cells)
        E_potential = E_cell.sum()
        
        acceleration = (positions - 2 * prev_positions + prev_prev_positions) / (dt ** 2)
        accel_penalty = 0.5*(acceleration ** 2).sum()
        #print(accel_penalty/E_potential)
        total_loss = E_potential + accel_penalty

        total_loss.backward()

        for p in optimizer.param_groups[0]['params']:
            if p.grad is not None and not p.grad.is_contiguous():
                p.grad = p.grad.contiguous()
        return total_loss
    
    optimizer.step(closure)
    
    trajectory.append(positions.detach().cpu().numpy())
    prev_prev_positions = prev_positions.clone().detach()
    prev_positions = positions.clone().detach()



In [8]:
total_pos = []
step = 0
print(len(trajectory))
for traj in trajectory:
    x = torch.tensor(traj, requires_grad=True, dtype=torch.float32, device=device)
    x_centers = x.view(20, 2)
    full = make_periodic_copies(x_centers)
    for site_index in range(full.shape[0]):
            total_pos.append([step, site_index, full[site_index][0][0].item(), full[site_index][0][1].item()])
    step += 1
trajectories_df = pd.DataFrame(total_pos, columns=['step', 'site_index', 'x', 'y'])
trajectories_df.to_csv("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/GNN for energy/GNN_20cells_8knn/trajectories_20.csv", index=False)

201


In [ ]:
x0_tensor = torch.tensor(x0.reshape(20,2), device=device, dtype=torch.float32)
x0_full = make_periodic_copies(x0_tensor).squeeze(1).detach().cpu().numpy()
df_initial = pd.DataFrame(x0_full, columns=['x','y'])
df_initial.to_csv("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/GNN for energy/GNN_20cells_8knn/positions_initiales_periodicite_20.csv", index=False)

In [ ]:
final_x_full = make_periodic_copies(positions)
final_x_full_np = final_x_full.squeeze(1).detach().cpu().numpy() 
E_cell = model(final_x_full, n_cells)
print(E_cell,sum(E_cell))
df = pd.DataFrame(final_x_full_np, columns=['x', 'y'])
df.to_csv("/home/jeanlienhard/Documents/Cell_GNN_clear/Cell_GNN/GNN for energy/GNN_20cells_8knn/trajectories_20.csv", index=False)

tensor([1.3914, 0.7853, 1.1016, 0.2569, 2.1560, 0.4748, 0.4590, 2.6500, 0.6081,
        0.8185, 0.3956, 0.7900, 0.2609, 0.6504, 1.7485, 0.6599, 1.2697, 0.6639,
        3.6834, 0.3413], device='cuda:0', grad_fn=<SqueezeBackward1>) tensor(21.1650, device='cuda:0', grad_fn=<AddBackward0>)


In [11]:
def compute_polygon_area_and_perimeter(polygon):
    polygon = np.array(polygon)
    x = polygon[:, 0]
    y = polygon[:, 1]
    area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
    perimeter = np.sum(np.linalg.norm(np.roll(polygon, -1, axis=0) - polygon, axis=1))
    return area, perimeter

def vornoi_area_and_perimeter(vor,target_indices):
    areas = []
    perimeters = []

    for idx in target_indices:
        region_index = vor.point_region[idx]
        region = vor.regions[region_index]
        if -1 in region or len(region) == 0:
            areas.append(1e-10)
            perimeters.append(1e-10)
            continue
        polygon = [vor.vertices[i] for i in region]
        area, perimeter = compute_polygon_area_and_perimeter(polygon)
        areas.append(area)
        perimeters.append(perimeter)          
    return areas,perimeters

In [12]:
def voronoi_loss(output,target_indices,accelerations,target_areas= 0.2,masse = 0.1, dt = 0.05,perimeter_target = 1.58):
    areas = []
    perimeters = []
    vor = Voronoi(output.cpu().detach().numpy())
    area,perimeter = vornoi_area_and_perimeter(vor,target_indices)
    areas.append(area)
    perimeters.append(perimeter)
    areas_tensor = torch.tensor(np.array(areas), dtype=torch.float32).to(device)
    perimeters_tensor = torch.tensor(np.array(perimeters),dtype=torch.float32).to(device)
    # areas_tensor = torch.stack(areas)
    # perimeters_tensor = torch.stack(perimeters)
    physics_loss = 0.02*(target_areas - areas_tensor)**2 + 0.005*(perimeters_tensor-perimeter_target)**2#+1e-5*(target_areas/(areas_tensor))**2
    # kinetic_loss = torch.sum((0.5*masse*dt**2)*(accelerations/(dt**2))**2,dim=-1)
    # print(physics_loss,kinetic_loss)
    return (physics_loss).squeeze(0)#+kinetic_loss

In [13]:
target_indices = np.arange(20)
print(1e4*voronoi_loss(final_x_full.view(500,2),target_indices,None))

tensor([0.4649, 3.0825, 4.0027, 0.5436, 4.9501, 1.1442, 0.4035, 6.2428, 1.9582,
        0.5790, 0.1728, 0.7456, 1.1154, 0.2941, 3.0629, 1.2981, 2.3574, 3.2066,
        6.4218, 0.3043], device='cuda:0')
